In [1]:
import time
import requests
import pandas as pd
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

API_KEY = "7b9a77a6449ac6e27a6e9b6c54b1d7b6"   # same GIE key as AGSI
URL = "https://alsi.gie.eu/api"               # ALSI = LNG terminals

# Session with automatic retries on timeouts / server errors
s = requests.Session()
s.headers.update({"x-key": API_KEY})
s.mount("https://", HTTPAdapter(max_retries=Retry(
    total=5, backoff_factor=2, status_forcelist=[429, 500, 502, 503, 504],
    allowed_methods=["GET"],
)))

params = {
    "type": "eu",          # EU aggregate
    "from": "2000-01-01",  # data actually starts ~2012
    "to": "2026-10-10",
    "size": 60,
    "page": 1,
}

rows = []
while True:
    r = s.get(URL, params=params, timeout=120)
    r.raise_for_status()
    js = r.json()
    for d in js["data"]:
        rows.append({
            "gasDayStart": d.get("gasDayStart"),
            "sendOut": d.get("sendOut"),  # GWh/d
            "status": d.get("status"),
        })
    last = int(js.get("last_page", 1))
    print(f"page {params['page']}/{last}")
    if params["page"] >= last:
        break
    params["page"] += 1
    time.sleep(1)  # stay under the API rate limit

df = pd.DataFrame(rows)
df["sendOut"] = pd.to_numeric(df["sendOut"], errors="coerce")
df["gasDayStart"] = pd.to_datetime(df["gasDayStart"])
df = df.sort_values("gasDayStart").reset_index(drop=True)

# ALSI sometimes publishes an impossible send-out, usually on an estimated (status E) day.
# Example: 5 Oct 2026 is 103,637 GWh/d, while the days around it are about 3,500.
# Do not drop every E row. Most estimates are normal; status only means the figure is provisional.
# A day is bad when it is more than 5 times the median of the previous 30 days.
# shift(1) keeps the day being tested out of its own benchmark.
# Replace it with that median. Leaving the cell empty is also wrong: the 7-day sum would
# treat the day as zero send-out. The median is the normal level of the surrounding days.
# sendout_bad marks the value as ours, not the API's.
raw = df["sendOut"].copy()
recent = raw.rolling(30, min_periods=10).median().shift(1)
df["sendout_bad"] = raw > 5 * recent
print(df.loc[df["sendout_bad"], ["gasDayStart", "status"]].assign(api_value=raw, filled=recent))
df.loc[df["sendout_bad"], "sendOut"] = recent

print(df.tail())
df.to_csv("lng_sendout.csv", index=False)

page 1/90
page 2/90
page 3/90
page 4/90
page 5/90
page 6/90
page 7/90
page 8/90
page 9/90
page 10/90
page 11/90
page 12/90
page 13/90
page 14/90
page 15/90
page 16/90
page 17/90
page 18/90
page 19/90
page 20/90
page 21/90
page 22/90
page 23/90
page 24/90
page 25/90
page 26/90
page 27/90
page 28/90
page 29/90
page 30/90
page 31/90
page 32/90
page 33/90
page 34/90
page 35/90
page 36/90
page 37/90
page 38/90
page 39/90
page 40/90
page 41/90
page 42/90
page 43/90
page 44/90
page 45/90
page 46/90
page 47/90
page 48/90
page 49/90
page 50/90
page 51/90
page 52/90
page 53/90
page 54/90
page 55/90
page 56/90
page 57/90
page 58/90
page 59/90
page 60/90
page 61/90
page 62/90
page 63/90
page 64/90
page 65/90
page 66/90
page 67/90
page 68/90
page 69/90
page 70/90
page 71/90
page 72/90
page 73/90
page 74/90
page 75/90
page 76/90
page 77/90
page 78/90
page 79/90
page 80/90
page 81/90
page 82/90
page 83/90
page 84/90
page 85/90
page 86/90
page 87/90
page 88/90
page 89/90
page 90/90
     gasDayStart   